# Badge 3: Data Application Builders Workshop: Lesson 4: SiS Behind the Scenes

The resource monitor and the DABW003 check.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, swapped for `BADGE3_DABW_ADMIN` (ACCOUNTADMIN), `BADGE3_DABW_ROLE` (SYSADMIN) and `BADGE3_DABW_WH` (COMPUTE_WH).
- **SQL equivalent** replaces a UI-only step; **From screenshot** marks code the course shows only as an image; **Derived solution** marks challenge answers the course does not give.
- DORA checks are not run here; see `dora/dora_tests.ipynb`.

Prerequisite: run `dora/account_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "BADGE3_DABW_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "BADGE3_DABW_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "BADGE3_DABW_WH"      # stands in for the course's COMPUTE_WH
import ast, io
from snowflake.snowpark.context import get_active_session

CUSTOM = 'SMOOTHIES.PUBLIC."Custom Smoothie Order Form"'
PENDING = 'SMOOTHIES.PUBLIC."Pending Smoothie Orders"'


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result


def write_app(app, app_code):
    """SQL equivalent of editing the app in Snowsight's Streamlit editor: write streamlit_app.py into the app's live version."""
    ast.parse(app_code)  # the code must at least be valid Python
    path = f"snow://streamlit/{app}/versions/live/streamlit_app.py"
    get_active_session().file.put_stream(io.BytesIO(app_code.encode()), path, auto_compress=False, overwrite=True)
    print(f"wrote {len(app_code.splitlines())} lines to {path}")

In [ ]:
%%sql -r session_context
use role {{SYSADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
select current_role(), current_warehouse(), current_database(), current_schema();

## Page: 💰 Managing Credit Usage for Your SiS App

> **Course:** Navigate to ADMIN → Cost Management → Resource Monitors and set one up that limits credit usage to 4 credits per day for your Account (screenshot 2026011415202: name **Four_Daily_Credits**, credit quota 4, monitor type Account, resets daily, immediately; notify at 75%, suspend and notify at 85%, suspend immediately and notify at 95%).

> ⚠️ **Note:** A monitor at the **account** level would cap every warehouse on this shared account, so it is attached to the stand-in warehouse instead.

In [ ]:
%%sql -r create_resource_monitor
-- SQL equivalent of the New Resource Monitor dialog (screenshot 2026011415202), as the real ACCOUNTADMIN.
-- The dialog's Monitor Type is Account; here the monitor is assigned to the stand-in warehouse instead.
use role accountadmin;
create resource monitor Four_Daily_Credits with
  credit_quota = 4
  frequency = daily
  start_timestamp = immediately
  triggers on 75 percent do notify
           on 85 percent do suspend
           on 95 percent do suspend_immediate;
alter warehouse {{WAREHOUSE}} set resource_monitor = Four_Daily_Credits;
-- restore context
use role {{SYSADMIN_ROLE}};
use warehouse {{WAREHOUSE}};

In [ ]:
%%sql -r check_resource_monitor
use role accountadmin;
show resource monitors like 'Four_Daily_Credits';
use role {{SYSADMIN_ROLE}};
use warehouse {{WAREHOUSE}};

## Page: 🐍 Importing Python Libraries

Reading only.

## Page: ⚠️ Exceeded My Quota? Wut?

Reading only (the page describes raising the monitor's daily credits when the quota message appears).

## Page: 🤖 DORA DABW003

### 🤖 DORA check DABW003 goes here
The course has the learner run **DABW003** (a mystery check: ASCII of the fruit name starting with Z) at this point.
It is in `dora/dora_tests.ipynb`, section **DABW003**, as a local check plus the exact DORA code.